# 04 - TinyValue Exercises

这一节是编程练习，不是继续讲概念。

目标是像课程作业一样，循序渐进地把 `Value` 的核心能力自己写出来：

```text
1. 先补局部代码
2. 再补一个方法
3. 最后写一个完整 TinyValue
```

练习格式统一为：

```text
留空代码 -> qa_check 测试 -> Show / Hide 提示 -> Show / Hide 答案
```

先自己写，卡住再看提示，最后才看答案。

<!-- xiao-preview:start -->
## 课前预习作业：开始写代码前先背下三条局部规则

这一本是作业本。先填 3 个小数字，确认你准备好写代码。


In [1]:
# TODO: 把 None 改成你手算出来的数字。
# 加法：out = x + y, out.grad = 4，x 收到多少梯度？
preview_add_x_grad = None

# 乘法：out = x * y, x=2, y=3, out.grad=4，x 收到多少梯度？
preview_mul_x_grad = None

# ReLU：out = relu(x), x=-2, out.grad=4，x 收到多少梯度？
preview_relu_neg_x_grad = None


def qa_check_04_preview(add_x_grad, mul_x_grad, relu_neg_x_grad):
    values = [add_x_grad, mul_x_grad, relu_neg_x_grad]
    if any(v is None for v in values):
        print('请先填写 TODO：先确认三条局部规则。')
        return
    assert add_x_grad == 4, '加法会把 out.grad 原样传给 x。'
    assert mul_x_grad == 12, f'乘法传给 x 的梯度应是 out.grad * y = 4 * 3 = 12，但你填的是 {mul_x_grad}'
    assert relu_neg_x_grad == 0, 'x < 0 时 ReLU 不传梯度。'
    print('OK: 预习通过，可以开始写 TinyValue。')


qa_check_04_preview(preview_add_x_grad, preview_mul_x_grad, preview_relu_neg_x_grad)


请先填写 TODO：先确认三条局部规则。


<!-- xiao-preview:hint -->
<details>
<summary><strong>Show / Hide 提示</strong></summary>

加法：原样传。乘法：乘另一个输入。ReLU：输入小于 0 时梯度断掉。

</details>


<!-- xiao-preview:answer -->
<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
preview_add_x_grad = 4
preview_mul_x_grad = 12
preview_relu_neg_x_grad = 0
```

</details>


## 作业模式 vs 参考答案模式

```text
作业模式：先运行留空代码，看到 TODO 提示；然后自己补代码，再运行 qa_check。
提示模式：只展开 Show / Hide 提示，不看完整答案。
答案模式：最后才展开 Show / Hide 答案，对照自己的实现。
```

不要一口气照着完整答案写完。micrograd 的实现就是很多小局部规则拼起来的。

## 直觉地图：局部反传规则

| 运算 | 前向 | 局部导数 | `_backward` 里要做什么 |
|---|---|---|---|
| 加法 | `out = x + y` | `dout/dx=1`, `dout/dy=1` | 把 `out.grad` 原样加给两边 |
| 乘法 | `out = x * y` | `dout/dx=y`, `dout/dy=x` | 交叉乘以后再加给两边 |
| 幂 | `out = x**n` | `dout/dx=n*x**(n-1)` | 乘上 `out.grad` 加回 x |
| ReLU | `max(0,x)` | 正数是 1，负数是 0 | 正数传梯度，负数不传 |
| backward | 从 L 出发 | 链式法则 | topo 倒序调用每个 `_backward` |

## 公式速记

这一节写代码时只用到这几条局部导数：

$$
out = x + y \Rightarrow \frac{\partial out}{\partial x}=1, \frac{\partial out}{\partial y}=1
$$

$$
out = xy \Rightarrow \frac{\partial out}{\partial x}=y, \frac{\partial out}{\partial y}=x
$$

$$
out = x^n \Rightarrow \frac{\partial out}{\partial x}=n x^{n-1}
$$

$$
ReLU(x)=\max(0,x)
$$

```text
x > 0 时，ReLU 局部导数是 1
x < 0 时，ReLU 局部导数是 0
```

代码里的 `_backward` 就是在把这些局部导数乘上 `out.grad`。

## 0. Setup

In [2]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd / 'micrograd', cwd.parent / 'micrograd']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'micrograd' / 'engine.py').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError(f'Could not find local micrograd package from {cwd}')

sys.path.insert(0, str(PROJECT_ROOT))
from micrograd.engine import Value


def assert_close(actual, expected, name):
    assert abs(actual - expected) < 1e-9, f'{name}: expected {expected}, got {actual}'


def _todo_if_zero_grad(grads, message):
    if all(abs(g) < 1e-12 for g in grads):
        print(message)
        return True
    return False


def qa_check_add_only(Cls):
    a = Cls(2.0)
    b = Cls(3.0)
    L = a + b
    L.backward()
    if _todo_if_zero_grad([a.grad, b.grad], 'TODO: 加法反向传播还没实现，a.grad / b.grad 还是 0。'):
        return
    assert_close(L.data, 5.0, 'L.data')
    assert_close(a.grad, 1.0, 'a.grad')
    assert_close(b.grad, 1.0, 'b.grad')
    print('OK: add-only passed.')


def qa_check_mul_only(Cls):
    a = Cls(2.0)
    b = Cls(3.0)
    L = a * b
    L.backward()
    if _todo_if_zero_grad([a.grad, b.grad], 'TODO: 乘法反向传播还没实现，a.grad / b.grad 还是 0。'):
        return
    assert_close(L.data, 6.0, 'L.data')
    assert_close(a.grad, 3.0, 'a.grad')
    assert_close(b.grad, 2.0, 'b.grad')
    print('OK: mul-only passed.')


def qa_check_pow_only(Cls):
    a = Cls(3.0)
    L = a ** 2
    L.backward()
    if _todo_if_zero_grad([a.grad], 'TODO: __pow__ 的反向传播还没实现，a.grad 还是 0。'):
        return
    assert_close(L.data, 9.0, 'L.data')
    assert_close(a.grad, 6.0, 'a.grad')
    print('OK: pow-only passed.')


def qa_check_relu_only(Cls):
    positive = Cls(2.0)
    negative = Cls(-2.0)
    L1 = positive.relu()
    L2 = negative.relu()
    L1.backward()
    L2.backward()
    if _todo_if_zero_grad([positive.grad], 'TODO: relu 正数侧还没传梯度，positive.grad 还是 0。'):
        return
    assert_close(L1.data, 2.0, 'relu positive data')
    assert_close(positive.grad, 1.0, 'relu positive grad')
    assert_close(L2.data, 0.0, 'relu negative data')
    assert_close(negative.grad, 0.0, 'relu negative grad')
    print('OK: relu-only passed.')


def qa_check_backward_topo(Cls):
    a = Cls(2.0)
    b = Cls(3.0)
    L = (a * b + a) * 2
    L.backward()
    if _todo_if_zero_grad([a.grad, b.grad], 'TODO: backward() 还没真正执行拓扑反传，a.grad / b.grad 还是 0。'):
        return
    assert_close(L.data, 16.0, 'L.data')
    assert_close(a.grad, 8.0, 'a.grad')
    assert_close(b.grad, 4.0, 'b.grad')
    print('OK: backward topo passed.')


def grade_value_class(Cls):
    # addition
    a = Cls(2.0)
    b = Cls(3.0)
    L = a + b
    L.backward()
    assert_close(L.data, 5.0, 'add L.data')
    assert_close(a.grad, 1.0, 'add a.grad')
    assert_close(b.grad, 1.0, 'add b.grad')

    # multiplication
    a = Cls(2.0)
    b = Cls(3.0)
    L = a * b
    L.backward()
    assert_close(L.data, 6.0, 'mul L.data')
    assert_close(a.grad, 3.0, 'mul a.grad')
    assert_close(b.grad, 2.0, 'mul b.grad')

    # repeated path
    a = Cls(2.0)
    L = a + a
    L.backward()
    assert_close(a.grad, 2.0, 'repeated path a.grad')

    # full expression
    a = Cls(2.0)
    b = Cls(3.0)
    L = 2 * (a * b + a)
    L.backward()
    assert_close(L.data, 16.0, 'full L.data')
    assert_close(a.grad, 8.0, 'full a.grad')
    assert_close(b.grad, 4.0, 'full b.grad')

    # power + relu
    a = Cls(2.0)
    b = Cls(3.0)
    L = (a * b + 1) ** 2
    L.backward()
    assert_close(L.data, 49.0, 'pow L.data')
    assert_close(a.grad, 42.0, 'pow a.grad')
    assert_close(b.grad, 28.0, 'pow b.grad')

    a = Cls(2.0)
    b = Cls(-3.0)
    L = (a * b + a).relu()
    L.backward()
    assert_close(L.data, 0.0, 'relu negative L.data')
    assert_close(a.grad, 0.0, 'relu negative a.grad')
    assert_close(b.grad, 0.0, 'relu negative b.grad')

    print('OK: grade_value_class passed.')


def qa_try_grade_value_class(Cls):
    try:
        grade_value_class(Cls)
    except (NotImplementedError, TypeError, AttributeError) as exc:
        print('TODO: 你的 TinyValue 还没有完整实现。当前卡在：')
        print(type(exc).__name__ + ':', exc)


## Exercise 1 - 补全加法的反向传播

题目：补全 `__add__` 里的 `_backward`。

In [3]:
class TinyValueAddOnly:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, TinyValueAddOnly) else TinyValueAddOnly(other)
        out = TinyValueAddOnly(self.data + other.data, (self, other), '+')

        def _backward():
            # TODO: 把 out.grad 传给 self 和 other
            pass

        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        self._backward()


qa_check_add_only(TinyValueAddOnly)

TODO: 加法反向传播还没实现，a.grad / b.grad 还是 0。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

加法 `out = self + other`，两个局部导数都是 1。所以 `out.grad` 应该原样传给 `self.grad` 和 `other.grad`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def _backward():
    self.grad += out.grad
    other.grad += out.grad
```

</details>

## Exercise 2 - 补全乘法的反向传播

题目：补全 `__mul__` 里的 `_backward`。

In [4]:
class TinyValueMulOnly:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __mul__(self, other):
        other = other if isinstance(other, TinyValueMulOnly) else TinyValueMulOnly(other)
        out = TinyValueMulOnly(self.data * other.data, (self, other), '*')

        def _backward():
            # TODO: 链式法则，把 out.grad 乘上局部导数
            pass

        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        self._backward()


qa_check_mul_only(TinyValueMulOnly)

TODO: 乘法反向传播还没实现，a.grad / b.grad 还是 0。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

`out = self * other`，所以 `dout/dself = other.data`，`dout/dother = self.data`。别忘了再乘上 `out.grad`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def _backward():
    self.grad += other.data * out.grad
    other.grad += self.data * out.grad
```

</details>

## Exercise 3 - 补全 `__pow__`

题目：实现幂运算，只支持数字次方。

In [5]:
class TinyValuePowOnly:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = TinyValuePowOnly(self.data ** other, (self,), f'**{other}')

        def _backward():
            # TODO: self.grad += ?
            pass

        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        self._backward()


qa_check_pow_only(TinyValuePowOnly)

TODO: __pow__ 的反向传播还没实现，a.grad 还是 0。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

公式是：如果 `out = x ** n`，那么 `dout/dx = n * x ** (n-1)`。代码里 `x` 是 `self.data`，`n` 是 `other`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def _backward():
    self.grad += (other * self.data ** (other - 1)) * out.grad
```

</details>

## Exercise 4 - 补全 `relu`

题目：实现 ReLU 的反向传播。

In [6]:
class TinyValueReluOnly:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def relu(self):
        out = TinyValueReluOnly(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            # TODO: 根据 out.data 是否大于 0 传梯度
            pass

        out._backward = _backward
        return out

    def backward(self):
        self.grad = 1
        self._backward()


qa_check_relu_only(TinyValueReluOnly)

TODO: relu 正数侧还没传梯度，positive.grad 还是 0。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

ReLU 正数侧导数是 1，负数侧导数是 0。可以用 `(out.data > 0)` 得到 `True/False`，在 Python 里会当作 `1/0` 用。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def _backward():
    self.grad += (out.data > 0) * out.grad
```

</details>

## Exercise 5 - 补全完整 `backward`

题目：实现完整反向传播。

步骤：

```text
1. 从最终输出开始 DFS
2. 先递归父节点，再 append 自己，得到 topo
3. 设置最终输出 self.grad = 1
4. reversed(topo)，依次调用每个节点的 _backward
```

In [7]:
class TinyValueTopoOnly:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, TinyValueTopoOnly) else TinyValueTopoOnly(other)
        out = TinyValueTopoOnly(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, TinyValueTopoOnly) else TinyValueTopoOnly(other)
        out = TinyValueTopoOnly(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            # TODO: 递归父节点，最后 append 自己
            pass

        # TODO: build_topo(self)
        # TODO: self.grad = 1
        # TODO: reversed(topo) 调用每个节点的 _backward

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other


qa_check_backward_topo(TinyValueTopoOnly)

TODO: backward() 还没真正执行拓扑反传，a.grad / b.grad 还是 0。


<details>
<summary><strong>Show / Hide 提示</strong></summary>

拓扑排序的关键是“先处理父节点，再 append 自己”。反向传播时要倒着执行：`for v in reversed(topo): v._backward()`。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
def backward(self):
    topo = []
    visited = set()

    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build_topo(child)
            topo.append(v)

    build_topo(self)
    self.grad = 1
    for v in reversed(topo):
        v._backward()
```

</details>

## Exercise 6 - 写完整 `MyTinyValue`

现在把前面 5 个练习合到一个类里。

先补下面的 `MyTinyValue`，再运行 `qa_try_grade_value_class(MyTinyValue)`。

In [8]:
class MyTinyValue:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        # TODO
        raise NotImplementedError

    def __mul__(self, other):
        # TODO
        raise NotImplementedError

    def __pow__(self, other):
        # TODO
        raise NotImplementedError

    def relu(self):
        # TODO
        raise NotImplementedError

    def backward(self):
        # TODO
        raise NotImplementedError

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def __repr__(self):
        return f'MyTinyValue(data={self.data}, grad={self.grad})'


qa_try_grade_value_class(MyTinyValue)

TODO: 你的 TinyValue 还没有完整实现。当前卡在：
NotImplementedError: 


<details>
<summary><strong>Show / Hide 提示</strong></summary>

可以把前面 5 个练习的答案拼起来。注意完整类还需要 `__radd__`、`__rmul__`，这样 `2 * value`、`value + 1` 这类表达式才能工作。

</details>

<details>
<summary><strong>Show / Hide 答案</strong></summary>

```python
class TinyValue:
    def __init__(self, data, children=(), op=''):
        self.data = data
        self.grad = 0
        self._prev = set(children)
        self._op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, TinyValue) else TinyValue(other)
        out = TinyValue(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, TinyValue) else TinyValue(other)
        out = TinyValue(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = TinyValue(self.data ** other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = TinyValue(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad

        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other**-1

    def __rtruediv__(self, other):
        return other * self**-1

    def __repr__(self):
        return f'TinyValue(data={self.data}, grad={self.grad})'
```

</details>

## Predict / Modify Lab

在看答案前，先预测这几个结果：

```text
1. L = a + a，a.grad 应该是多少？为什么需要 +=？
2. L = a * b，a=4, b=5 时，a.grad 和 b.grad 分别是多少？
3. ReLU 输入是 -2 时，前面的梯度会不会继续传？
4. 如果忘了 self.grad = 1，所有叶子节点的 grad 会发生什么？
```

改一行实验：

```text
1. 把 Exercise 1 的 a=2,b=3 改成 a=10,b=20，观察加法梯度是否变化。
2. 把 Exercise 2 的 a=2,b=3 改成 a=4,b=5，先预测再运行。
3. 把完整实现里的 += 改成 =，再测试 L = a + a。
```

这些小实验比背代码更重要。

## What To Remember

这一节过关，只需要能讲清楚这 5 句话：

```text
1. 每个运算都要做前向：创建 out。
2. 每个运算都要给 out 绑定一个局部 _backward。
3. _backward 里用 +=，因为一个变量可能有多条路径。
4. backward() 要先 topo 排序，再倒序执行。
5. 一个完整 TinyValue 其实就是这些小规则拼起来。
```

<!-- xiao-post:start -->
## 课后作业提交清单

这一节学完后，用下面 5 条自检：

```text
1. 我独立补全了加法、乘法、平方、ReLU 的局部反传。
2. 我能写出 backward 的拓扑排序。
3. 我的 MyTinyValue 通过 grade_value_class。
4. 我能解释每个 qa_check 抓的是什么 bug。
5. 我能不看答案写出一个最小 TinyValue。
```


## AI 复盘检查 Prompt

把下面这段发送给任意 AI，让它检查你是否能自己实现 TinyValue：

```text
你是我的 micrograd 编程练习检查官。

我刚学完一节内容，主题是：自己实现一个简化版 TinyValue，支持 +、*、**、relu 和 backward。

请你用一问一答的方式检查我。规则：
1. 一次只问一个问题，等我回答后再评价。
2. 优先让我补代码，不要直接给完整答案。
3. 如果我卡住，先给提示，再给局部代码，不要一次性给全量实现。
4. 请重点检查我是否理解 +=、out.grad、局部导数、topo、reversed(topo)。
5. 最后给我一个 0-100 的掌握度评分，并告诉我是否可以进入 Neuron / MLP。

现在从第 1 个问题开始问我。
```